# QuickPCA Quickstart

**Essential-dynamics PCA, Free-Energy Landscapes and a publication-ready report in a few lines of Python.**

QuickPCA distils a molecular-dynamics (MD) trajectory — potentially millions of
atomic coordinates — down to the handful of collective motions that matter. This
notebook walks through the complete pipeline on the bundled sample data:

1. **Load** a trajectory with `load_trajectory`.
2. **Decompose** the motion with `compute_pca` (Kabsch alignment + SVD-based PCA).
3. **Map the energetics** with `compute_fel` (a Boltzmann-inverted free-energy landscape).
4. **Visualise everything** with `plot_report`, shown inline below.

> QuickPCA was developed by **Gleb Novikov**. It is MIT licensed.

## 0. Setup

We import the public API and silence a few noisy third-party warnings so the
output stays readable. Everything below uses only the documented public
functions:

```python
from quickpca import (
    compute_pca, compute_pca_from_files, compute_fel,
    plot_report, load_trajectory, get_backend,
)
```

In [ ]:
import warnings

warnings.filterwarnings("ignore")  # quieten MDAnalysis / numpy deprecation chatter

from quickpca import (
    __version__,
    compute_fel,
    compute_pca,
    load_trajectory,
    plot_report,
)

print(f"QuickPCA version: {__version__}")

## 1. Load the sample trajectory

The repository ships with a small MD dataset under `../data/`:

- `reference.pdb` — the topology (atom names, residues, connectivity).
- `trajectory.nc` — an AMBER NetCDF trajectory of the same system over time.

`load_trajectory` uses [MDAnalysis](https://www.mdanalysis.org/) under the hood.
Two arguments keep things fast and focused:

- `selection="name CA"` restricts the analysis to backbone **C-alpha** atoms —
  one representative atom per residue, the standard choice for capturing a
  protein's large-scale "essential" motions.
- `interval=10` keeps every 10th frame (a *stride*). Striding keeps this demo
  snappy; drop it to `1` for a production run.

The returned `Trajectory` is a simple dataclass: `coords` has shape
`(n_frames, n_atoms, 3)` plus per-atom metadata (`resids`, `resnames`, ...).

In [ ]:
traj = load_trajectory(
    "../data/reference.pdb",
    "../data/trajectory.nc",
    selection="name CA",
    interval=10,  # stride: keep every 10th frame to keep the demo quick
)

print(f"frames      : {traj.n_frames}")
print(f"atoms (CA)  : {traj.n_atoms}")
print(f"coords shape: {traj.coords.shape}  (frames, atoms, xyz)")
print(f"residues    : {traj.resids.min()}–{traj.resids.max()}")

## 2. Run the PCA

`compute_pca` performs the full essential-dynamics pipeline:

1. **Kabsch alignment** — every frame is rigid-body superimposed onto a reference
   frame, so we measure *internal* conformational change rather than overall
   translation/rotation of the molecule.
2. **PCA via SVD** — the `(n_frames, 3·n_atoms)` displacement matrix is decomposed
   directly with a singular-value decomposition. This is numerically equivalent to
   diagonalising the covariance matrix but faster and more stable.

The resulting `PCAResult` carries, among other things:

| Attribute | Meaning |
|---|---|
| `projections` | each frame's coordinates in PC space, shape `(n_frames, n_components)` |
| `explained_variance_ratio` | fraction of total motion captured by each PC |
| `cumulative_variance` | running total of the above |
| `components` | the eigenvectors (the actual collective motions) |
| `cross_correlation` | residue–residue motion correlation matrix |

We pass `backend="numpy"` here; notebook **02** shows how to swap in a JAX backend
for GPU/TPU acceleration.

In [ ]:
pca = compute_pca(
    traj.coords,
    n_components=10,
    backend="numpy",
    align=True,
)

print(f"backend           : {pca.backend}")
print(f"components kept    : {pca.n_components}")
print(f"projections shape  : {pca.projections.shape}")
print(f"cross-corr shape   : {pca.cross_correlation.shape}")

### How much motion does each PC capture?

A defining feature of essential dynamics is that a *few* components usually
explain *most* of the motion. Let's print the explained-variance ratio and the
cumulative total — this is the quantitative justification for reducing a
high-dimensional trajectory down to a 2-D PC1/PC2 picture.

In [ ]:
evr = pca.explained_variance_ratio * 100.0
cum = pca.cumulative_variance * 100.0

print("PC   variance%   cumulative%")
print("-" * 30)
for i in range(pca.n_components):
    print(f"{i + 1:>2}   {evr[i]:7.2f}    {cum[i]:8.2f}")

print()
print(f"PC1 + PC2 capture {cum[1]:.1f}% of all the (aligned) motion.")

## 3. Build the Free-Energy Landscape (FEL)

The PC1/PC2 projections form a 2-D cloud of sampled conformations. By histogramming
that cloud and applying a **Boltzmann inversion** we turn *how often* a region is
visited into *how energetically favourable* it is:

$$ F(\mathrm{PC1},\,\mathrm{PC2}) = -k_\mathrm{B}T \,\ln \rho(\mathrm{PC1},\,\mathrm{PC2}) $$

where $\rho$ is the (Gaussian-smoothed) probability density. Frequently visited
states become **low-energy basins** (minima); rarely visited transition regions
become **barriers**. The landscape is shifted so that its global minimum sits at
$F = 0$.

`compute_fel` exposes the key physical and numerical knobs:

- `temperature` (K) — sets the energy scale $k_\mathrm{B}T$.
- `n_bins` — resolution of the 2-D histogram.
- `sigma` — Gaussian smoothing width (in bins).

Notebook **03** is a dedicated deep-dive on these parameters.

In [ ]:
fel = compute_fel(
    pca.projections,
    temperature=300.0,  # K — physiological-ish; sets the kT energy scale
    n_bins=50,
    sigma=1.0,
)

import numpy as np

print(f"temperature : {fel.temperature:.0f} K")
print(f"kBT         : {fel.kBT:.3f} kJ/mol")
print(f"F grid      : {fel.F.shape}  (n_bins x n_bins)")
print(f"F range     : {np.nanmin(fel.F):.2f} – {np.nanmax(fel.F):.2f} kJ/mol")

## 4. Render the report

`plot_report` assembles a single publication-ready figure with four panels:

- **Top-left** — the Free-Energy Landscape (PC1 vs PC2) with the trajectory path
  overlaid; green/red stars mark the first/last frame.
- **Top-right** — the residue cross-correlation matrix (which parts of the
  molecule move together, in red, or in opposition, in blue).
- **Bottom-left** — the explained-variance bar chart with a cumulative overlay.
- **Bottom-right** — PC1/PC2 projection distributions (histogram + KDE).

The function saves a PNG and returns its path. We write it next to the notebook
and then display it inline.

In [ ]:
report_path = plot_report(
    pca,
    fel,
    output="quickstart_report.png",
    temperature=300.0,
    title="QuickPCA Quickstart — Essential Dynamics Report",
)
print(f"saved: {report_path}")

In [ ]:
from IPython.display import Image, display

display(Image(filename=report_path))

## 5. Reading the report

- **Free-Energy Landscape** — look for the deep-blue **basins**: these are the
  stable conformational states the molecule actually populates. A single broad
  basin means one dominant state; multiple basins separated by warmer (red)
  barriers indicate distinct states with transitions between them. The white path
  traces how the simulation moved through these states over time.
- **Cross-correlation matrix** — red blocks off the diagonal flag groups of
  residues that move *together* (e.g. a rigid domain); blue blocks flag
  *anti-correlated* motion (e.g. a hinge opening and closing).
- **Explained-variance chart** — confirms that the first one or two PCs dominate,
  which is what justifies summarising the dynamics in a 2-D landscape.
- **Projection distributions** — multi-modal (multi-peak) histograms are another
  signature of distinct conformational states along a given PC.

## One-liner: from files straight to a result

If you just want a `PCAResult` and don't need the intermediate `Trajectory`,
`compute_pca_from_files` fuses loading and PCA into one call.

In [ ]:
from quickpca import compute_pca_from_files

pca_direct = compute_pca_from_files(
    "../data/reference.pdb",
    "../data/trajectory.nc",
    selection="name CA",
    interval=10,
    n_components=10,
    backend="numpy",
)

# Identical setup, so the variance profile matches what we computed above.
agree = np.allclose(
    pca_direct.explained_variance_ratio,
    pca.explained_variance_ratio,
)
print(f"one-liner reproduces the step-by-step result: {agree}")

## Where to next?

- **`02_jax_gpu.ipynb`** — run the exact same pipeline on the JAX backend for
  GPU/TPU acceleration, and confirm the numbers agree with NumPy.
- **`03_free_energy_landscape.ipynb`** — a deep dive into `compute_fel`: Boltzmann
  inversion, temperature, binning, smoothing, and how to read minima and basins.

---
*QuickPCA — developed by Gleb Novikov. MIT licensed.*